In [ ]:
# NLP Session 1 — Customer Message Preprocessing

This notebook follows the complete assignment using the same CSV dataset for every task.

In [1]:
# Imports and setup
import re
import pandas as pd
import nltk
import spacy
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
from collections import Counter
from textblob import TextBlob

DATA_PATH = r"D:\mahmoud\My_Courses\R5_Machine_Learning_Microsoft_AI_Data_Science\ENG Baraa\NLP_Processing\Task_Data_Set\nlp_customer_messages - nlp_customer_messages.csv"

# Download NLTK resources safely
for package in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.data.find(package if package != "punkt_tab" else "tokenizers/punkt_tab")
    except LookupError:
        nltk.download(package, quiet=True)

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("Libraries and resources are ready.")

Libraries and resources are ready.


In [2]:
# Task 1: load and inspect the CSV
import os
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"CSV not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df["char_count"] = df["message"].astype(str).str.len()
print("First 5 messages:")
display(df[["message_id", "message"]].head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nMissing values:")
print(df.isna().sum())
print("Duplicate message_id values:", int(df["message_id"].duplicated().sum()))
print("Longest message:\n", df.loc[df["char_count"].idxmax(), "message"])
print("Shortest message:\n", df.loc[df["char_count"].idxmin(), "message"])
print("\nchar_count measures the number of characters, including spaces, in each original message.")

First 5 messages:


,message_id,message
0,MSG001,"Good morning, My phone is not working :( Than..."
1,MSG002,"Hi, Thanks for your quick response! I need hel..."
2,MSG003,"Hi support, I LOVE this product 😍😍 Can someone..."
3,MSG004,"Hey there, The new update is AMAZING!! Please ..."
4,MSG005,"Dear team, The price is 250 EGP, too expensive..."


Shape: (80, 3)
Columns: ['message_id', 'message', 'char_count']

Missing values:
message_id    0
message       0
char_count    0
dtype: int64
Duplicate message_id values: 0
Longest message:
 Hey there, I need help with payment for order 10612. Please check. Email: customer36@example.com Visit https://help.example.com/ticket/10612   Please   help   me.   My backup email is user.name+test@sample.org
Shortest message:
 Dear team, Thanks! Great service :) Thank you.

char_count measures the number of characters, including spaces, in each original message.


In [ ]:
## Part A — Data Understanding and Regex
### Task 2 — Find and extract customer-message patterns

In [3]:
# Regex extraction patterns and reusable functions
EMAIL_PATTERN = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
URL_PATTERN = re.compile(r"https?://[^\s]+", re.IGNORECASE)
PHONE_PATTERN = re.compile(r"010\d{8}")
NUMBER_PATTERN = re.compile(r"\d+")

def extract_email(text): return EMAIL_PATTERN.findall(str(text))
def extract_url(text): return [u.rstrip(".,);]}") for u in URL_PATTERN.findall(str(text))]
def extract_phone(text): return PHONE_PATTERN.findall(str(text))
def extract_numbers(text): return NUMBER_PATTERN.findall(str(text))
def has_urgent(text): return bool(re.search(r"\burgent\b", str(text), re.IGNORECASE))

df["emails"] = df["message"].apply(extract_email)
df["urls"] = df["message"].apply(extract_url)
df["phones"] = df["message"].apply(extract_phone)
df["numbers"] = df["message"].apply(extract_numbers)
df["has_urgent"] = df["message"].apply(has_urgent)

cols = ["message_id", "message", "emails", "urls", "phones", "numbers", "has_urgent"]
mask = df[["emails", "urls", "phones", "has_urgent"]].apply(lambda s: s.astype(bool)).any(axis=1)
display(df.loc[mask, cols].head(10))
print("Messages with email:", int(df["emails"].apply(bool).sum()))
print("Messages with phone:", int(df["phones"].apply(bool).sum()))
print("Messages with URL:", int(df["urls"].apply(bool).sum()))
print("Messages with urgent:", int(df["has_urgent"].sum()))

,message_id,message,emails,urls,phones,numbers,has_urgent
2,MSG003,"Hi support, I LOVE this product 😍😍 Can someone...",[customer3@example.com],[],[],[3],False
3,MSG004,"Hey there, The new update is AMAZING!! Please ...",[],[https://help.example.com/ticket/10068],[],[10068],False
4,MSG005,"Dear team, The price is 250 EGP, too expensive...",[],[],[01010011705],"[250, 01010011705]",False
5,MSG006,"Dear support, The package for order 10102 is d...",[customer6@example.com],[],[],"[10102, 6]",False
6,MSG007,"Hello, Bad quality... very disappointed. What ...",[],[],[],[],True
7,MSG008,"Hello team, Great quality, but delivery took 7...",[],[https://help.example.com/ticket/10136],[],"[7, 10136]",False
8,MSG009,"Good morning, Thanks! Great service :) Thanks ...","[customer9@example.com, user.name+test@sample....",[],[],[9],False
9,MSG010,"Hi, Can you change my delivery address? I need...",[],[],[01010023410],[01010023410],False
11,MSG012,"Hey there, I cannot log in to my account. Plea...",[customer12@example.com],[https://help.example.com/ticket/10204],[],"[12, 10204]",False
13,MSG014,"Dear support, The app crashes every time I ope...",[],[],[],[],True


Messages with email: 26
Messages with phone: 16
Messages with URL: 20
Messages with urgent: 11


In [4]:
# Task 3: regex cleaning
# Specific patterns are replaced first so numbers inside them are not cleaned again.
def clean_text(text):
    text = str(text)
    text = EMAIL_PATTERN.sub("[EMAIL]", text)
    text = URL_PATTERN.sub("[URL]", text)
    text = PHONE_PATTERN.sub("[PHONE]", text)
    text = NUMBER_PATTERN.sub("[NUMBER]", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_message"] = df["message"].apply(clean_text)
display(df[["message", "clean_message"]].head(8))

,message,clean_message
0,"Good morning, My phone is not working :( Than...","Good morning, My phone is not working :( Thank..."
1,"Hi, Thanks for your quick response! I need hel...","Hi, Thanks for your quick response! I need hel..."
2,"Hi support, I LOVE this product 😍😍 Can someone...","Hi support, I LOVE this product 😍😍 Can someone..."
3,"Hey there, The new update is AMAZING!! Please ...","Hey there, The new update is AMAZING!! Please ..."
4,"Dear team, The price is 250 EGP, too expensive...","Dear team, The price is [NUMBER] EGP, too expe..."
5,"Dear support, The package for order 10102 is d...","Dear support, The package for order [NUMBER] i..."
6,"Hello, Bad quality... very disappointed. What ...","Hello, Bad quality... very disappointed. What ..."
7,"Hello team, Great quality, but delivery took 7...","Hello team, Great quality, but delivery took [..."


In [6]:
# Task 4: test regex rules
regex_tests = [
    "Please email me at sara.test@example.com today.",
    "Call the customer on 01012345678, please.",
    "Visit https://help.example.com/tickets/42 for information.",
    "This ordinary message has no contact details."
]
for test in regex_tests:
    print({"text": test, "emails": extract_email(test), "phones": extract_phone(test),
           "urls": extract_url(test), "clean": clean_text(test)})

print("Pattern limitation: the phone pattern accepts only Egyptian-style 010 plus 8 digits and does not validate all international phone formats. URLs without http/https may also be missed.")
print("The patterns also use typical email formats and will miss less common valid email forms.")

{'text': 'Please email me at sara.test@example.com today.', 'emails': ['sara.test@example.com'], 'phones': [], 'urls': [], 'clean': 'Please email me at [EMAIL] today.'}
{'text': 'Call the customer on 01012345678, please.', 'emails': [], 'phones': ['01012345678'], 'urls': [], 'clean': 'Call the customer on [PHONE], please.'}
{'text': 'Visit https://help.example.com/tickets/42 for information.', 'emails': [], 'phones': [], 'urls': ['https://help.example.com/tickets/42'], 'clean': 'Visit [URL] for information.'}
{'text': 'This ordinary message has no contact details.', 'emails': [], 'phones': [], 'urls': [], 'clean': 'This ordinary message has no contact details.'}
Pattern limitation: the phone pattern accepts only Egyptian-style 010 plus 8 digits and does not validate all international phone formats. URLs without http/https may also be missed.
The patterns also use typical email formats and will miss less common valid email forms.


In [ ]:
## Part B — Tokenization
### Task 5 — NLTK sentence and word tokenization

In [7]:
# NLTK tokenization helpers

def nltk_sentences(text):
    return nltk.sent_tokenize(str(text))

def nltk_words(text):
    return nltk.word_tokenize(str(text))

df["sentences_nltk"] = df["clean_message"].apply(nltk_sentences)
df["tokens_nltk"] = df["clean_message"].apply(nltk_words)
df["sentence_count"] = df["sentences_nltk"].apply(len)
df["token_count"] = df["tokens_nltk"].apply(len)
display(df.nlargest(5, "token_count")[["message_id", "message", "token_count", "tokens_nltk"]])
print("Token count can differ from an informal word count because punctuation, URLs/placeholders, and contractions may each become separate tokens.")

,message_id,message,token_count,tokens_nltk
65,MSG066,"Hi, The package for order 11122 is damaged. I ...",38,"[Hi, ,, The, package, for, order, [, NUMBER, ]..."
35,MSG036,"Hey there, I need help with payment for order ...",37,"[Hey, there, ,, I, need, help, with, payment, ..."
71,MSG072,"Hello team, I cannot log in to my account. Ple...",36,"[Hello, team, ,, I, can, not, log, in, to, my,..."
44,MSG045,"Dear team, The price is 250 EGP, too expensive...",35,"[Dear, team, ,, The, price, is, [, NUMBER, ], ..."
59,MSG060,"Hey there, My order 11020 arrived late!!! Plea...",35,"[Hey, there, ,, My, order, [, NUMBER, ], arriv..."


Token count can differ from an informal word count because punctuation, URLs/placeholders, and contractions may each become separate tokens.


### Task 6 — spaCy tokenization and comparison

In [8]:
# spaCy tokenization
df["tokens_spacy"] = [ [t.text for t in nlp(str(text))] for text in df["clean_message"] ]
comparison = pd.DataFrame({
    "message_id": df["message_id"].head(10),
    "clean_message": df["clean_message"].head(10),
    "nltk_tokens": df["tokens_nltk"].head(10),
    "spacy_tokens": df["tokens_spacy"].head(10)
})
display(comparison)

differences = comparison[comparison["nltk_tokens"].astype(str) != comparison["spacy_tokens"].astype(str)]
print("Messages with visible token differences:", len(differences))
display(differences.head())

sample_tokens = [(t.text, t.is_alpha, t.is_punct) for t in nlp(str(df.loc[0, "clean_message"]))]
print(pd.DataFrame(sample_tokens, columns=["text", "is_alpha", "is_punct"]))
print("NLTK and spaCy usually produce similar whitespace-based tokens, but punctuation and special forms can be split differently. Context-aware spaCy also provides token attributes such as is_alpha and is_punct.")

,message_id,clean_message,nltk_tokens,spacy_tokens
0,MSG001,"Good morning, My phone is not working :( Thank...","[Good, morning, ,, My, phone, is, not, working...","[Good, morning, ,, My, phone, is, not, working..."
1,MSG002,"Hi, Thanks for your quick response! I need hel...","[Hi, ,, Thanks, for, your, quick, response, !,...","[Hi, ,, Thanks, for, your, quick, response, !,..."
2,MSG003,"Hi support, I LOVE this product 😍😍 Can someone...","[Hi, support, ,, I, LOVE, this, product, 😍😍, C...","[Hi, support, ,, I, LOVE, this, product, 😍, 😍,..."
3,MSG004,"Hey there, The new update is AMAZING!! Please ...","[Hey, there, ,, The, new, update, is, AMAZING,...","[Hey, there, ,, The, new, update, is, AMAZING,..."
4,MSG005,"Dear team, The price is [NUMBER] EGP, too expe...","[Dear, team, ,, The, price, is, [, NUMBER, ], ...","[Dear, team, ,, The, price, is, [, NUMBER, ], ..."
5,MSG006,"Dear support, The package for order [NUMBER] i...","[Dear, support, ,, The, package, for, order, [...","[Dear, support, ,, The, package, for, order, [..."
6,MSG007,"Hello, Bad quality... very disappointed. What ...","[Hello, ,, Bad, quality, ..., very, disappoint...","[Hello, ,, Bad, quality, ..., very, disappoint..."
7,MSG008,"Hello team, Great quality, but delivery took [...","[Hello, team, ,, Great, quality, ,, but, deliv...","[Hello, team, ,, Great, quality, ,, but, deliv..."
8,MSG009,"Good morning, Thanks! Great service :) Thanks ...","[Good, morning, ,, Thanks, !, Great, service, ...","[Good, morning, ,, Thanks, !, Great, service, ..."
9,MSG010,"Hi, Can you change my delivery address? I need...","[Hi, ,, Can, you, change, my, delivery, addres...","[Hi, ,, Can, you, change, my, delivery, addres..."


Messages with visible token differences: 4


,message_id,clean_message,nltk_tokens,spacy_tokens
0,MSG001,"Good morning, My phone is not working :( Thank...","[Good, morning, ,, My, phone, is, not, working...","[Good, morning, ,, My, phone, is, not, working..."
2,MSG003,"Hi support, I LOVE this product 😍😍 Can someone...","[Hi, support, ,, I, LOVE, this, product, 😍😍, C...","[Hi, support, ,, I, LOVE, this, product, 😍, 😍,..."
6,MSG007,"Hello, Bad quality... very disappointed. What ...","[Hello, ,, Bad, quality, ..., very, disappoint...","[Hello, ,, Bad, quality, ..., very, disappoint..."
8,MSG009,"Good morning, Thanks! Great service :) Thanks ...","[Good, morning, ,, Thanks, !, Great, service, ...","[Good, morning, ,, Thanks, !, Great, service, ..."


       text  is_alpha  is_punct
0      Good      True     False
1   morning      True     False
2         ,     False      True
3        My      True     False
4     phone      True     False
5        is      True     False
6       not      True     False
7   working      True     False
8        :(     False      True
9    Thanks      True     False
10    again      True     False
11        !     False      True
NLTK and spaCy usually produce similar whitespace-based tokens, but punctuation and special forms can be split differently. Context-aware spaCy also provides token attributes such as is_alpha and is_punct.


## Part C — Case Folding
### Task 7 — Lowercase and vocabulary size

In [9]:
df["lower_message"] = df["clean_message"].str.lower()
df["lower_tokens"] = df["lower_message"].apply(nltk_words)

case_sensitive_vocab = set()
for tokens in df["tokens_nltk"]:
    case_sensitive_vocab.update(tokens)

lower_vocab = set()
for tokens in df["lower_tokens"]:
    lower_vocab.update(tokens)

print("Unique case-sensitive tokens:", len(case_sensitive_vocab))
print("Unique tokens after lowercase:", len(lower_vocab))
print("Vocabulary reduction:", len(case_sensitive_vocab) - len(lower_vocab))

# Find a clear uppercase/lowercase pair in the tokenized dataset.
pairs = []
lower_to_original = {}
for token in case_sensitive_vocab:
    lower_to_original.setdefault(token.lower(), []).append(token)
for words in lower_to_original.values():
    if len(set(words)) > 1:
        pairs.append(sorted(set(words)))
        break
print("Example of case variation:", pairs[0] if pairs else "No different uppercase/lowercase form found.")
print("Lowercasing combines differently capitalized forms and therefore reduces vocabulary size. However, it can remove useful information such as emphasis (HELP!) or named/proper nouns in some tasks.")

Unique case-sensitive tokens: 122
Unique tokens after lowercase: 113
Vocabulary reduction: 9
Example of case variation: ['Can', 'can']
Lowercasing combines differently capitalized forms and therefore reduces vocabulary size. However, it can remove useful information such as emphasis (HELP!) or named/proper nouns in some tasks.


## Part D — Stemming and Lemmatization
### Task 8 — Stem with three algorithms

In [10]:
porter = PorterStemmer()
snowball = SnowballStemmer("english")
lancaster = LancasterStemmer()

def alpha_tokens(tokens):
    return [t.lower() for t in tokens if t.isalpha()]

df["porter_stems"] = df["lower_tokens"].apply(lambda ts: [porter.stem(t) for t in alpha_tokens(ts)])
df["snowball_stems"] = df["lower_tokens"].apply(lambda ts: [snowball.stem(t) for t in alpha_tokens(ts)])
df["lancaster_stems"] = df["lower_tokens"].apply(lambda ts: [lancaster.stem(t) for t in alpha_tokens(ts)])

# Select up to ten real alphabetic words from the dataset.
real_words = []
for tokens in df["lower_tokens"]:
    for token in tokens:
        if token.isalpha() and token not in real_words:
            real_words.append(token)
        if len(real_words) >= 10:
            break
    if len(real_words) >= 10:
        break
stem_table = pd.DataFrame({
    "original": real_words,
    "porter": [porter.stem(w) for w in real_words],
    "snowball": [snowball.stem(w) for w in real_words],
    "lancaster": [lancaster.stem(w) for w in real_words]
})
display(stem_table)
print("Example non-dictionary stem(s):", [w for w in stem_table["lancaster"].tolist() if w not in set(real_words)][:5])

,original,porter,snowball,lancaster
0,good,good,good,good
1,morning,morn,morn,morn
2,my,my,my,my
3,phone,phone,phone,phon
4,is,is,is,is
5,not,not,not,not
6,working,work,work,work
7,thanks,thank,thank,thank
8,again,again,again,again
9,hi,hi,hi,hi


Example non-dictionary stem(s): ['morn', 'phon', 'work', 'thank']


### Task 9 — NLTK and spaCy lemmatization

In [11]:
lemmatizer = WordNetLemmatizer()
df["lemmas_nltk"] = df["lower_tokens"].apply(lambda ts: [lemmatizer.lemmatize(t) for t in alpha_tokens(ts)])
df["lemmas_spacy"] = [[t.lemma_.lower() for t in doc if t.is_alpha] for doc in nlp.pipe(df["clean_message"].astype(str), batch_size=50)]

compare_words = real_words[:10]
lemma_table = pd.DataFrame({
    "original": compare_words,
    "porter_stem": [porter.stem(w) for w in compare_words],
    "nltk_lemma": [lemmatizer.lemmatize(w) for w in compare_words],
    "spacy_lemma": [[t.lemma_.lower() for t in nlp(w) if t.is_alpha][0] for w in compare_words]
})
display(lemma_table)
different = lemma_table[lemma_table["porter_stem"] != lemma_table["nltk_lemma"]]
print("Stem/lemma differences found:", len(different))
print("spaCy uses context and POS information in readable text, while the default NLTK WordNetLemmatizer assumes noun unless POS is supplied.")

,original,porter_stem,nltk_lemma,spacy_lemma
0,good,good,good,good
1,morning,morn,morning,morning
2,my,my,my,my
3,phone,phone,phone,phone
4,is,is,is,be
5,not,not,not,not
6,working,work,working,work
7,thanks,thank,thanks,thank
8,again,again,again,again
9,hi,hi,hi,hi


Stem/lemma differences found: 3
spaCy uses context and POS information in readable text, while the default NLTK WordNetLemmatizer assumes noun unless POS is supplied.


## Part E — Stop Words
### Task 10 — Compare NLTK and spaCy stop-word removal

In [ ]:
nltk_stop = set(stopwords.words("english"))
spacy_stop = {word for word in nlp.vocab.strings if nlp.vocab[word].is_stop}

def remove_stops(tokens, stop_list):
    return [t for t in tokens if t.lower() not in stop_list]

df["tokens_no_stop_nltk"] = df["lower_tokens"].apply(lambda ts: remove_stops(ts, nltk_stop))
df["tokens_no_stop_spacy"] = df["lower_tokens"].apply(lambda ts: remove_stops(ts, spacy_stop))
stop_comparison = pd.DataFrame({
    "tokens_before": df["lower_tokens"].head(10).apply(len),
    "nltk_after": df["tokens_no_stop_nltk"].head(10).apply(len),
    "spacy_after": df["tokens_no_stop_spacy"].head(10).apply(len),
    "same_result": df.head(10).apply(lambda r: r["tokens_no_stop_nltk"] == r["tokens_no_stop_spacy"], axis=1)
})
display(stop_comparison)
print("Stop-word lists are exactly the same:", nltk_stop == spacy_stop)

negation_mask = df["lower_message"].str.contains(r"\b(?:not|no|never)\b", regex=True, na=False)
negation_examples = df.loc[negation_mask].head(1)
display(negation_examples[["message_id", "message", "tokens_no_stop_nltk"]])
kept_negations = {"not", "no", "never"}

def remove_stops_keep_negation(tokens):
    return [t for t in tokens if t.lower() not in nltk_stop or t.lower() in kept_negations]

if not negation_examples.empty:
    index = negation_examples.index[0]
    df["tokens_stop_keep_negation"] = df["lower_tokens"].apply(remove_stops_keep_negation)
    print("Normal NLTK policy:", df.loc[index, "tokens_no_stop_nltk"])
    print("Policy keeping negations:", df.loc[index, "tokens_stop_keep_negation"])
print("Removing negations can turn 'not good' or 'never works' into positive-neutral ideas. Sentiment systems should keep not, no, and never unless negation is handled explicitly.")

,tokens_before,nltk_after,spacy_after,same_result
0,13,9,9,True
1,15,12,12,True
2,18,14,13,False
3,17,14,13,False
4,23,17,16,False
5,27,21,20,False
6,17,12,12,True
7,22,21,20,False
8,24,21,21,True
9,22,16,15,False


Stop-word lists are exactly the same: False


C:\Users\PC\AppData\Local\Temp\ipykernel_11348\2005478077.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  negation_mask = df["lower_message"].str.contains(r"\b(not|no|never)\b", regex=True, na=False)


,message_id,message,tokens_no_stop_nltk
0,MSG001,"Good morning, My phone is not working :( Than...","[good, morning, ,, phone, working, :, (, thank..."


Normal NLTK policy: ['good', 'morning', ',', 'phone', 'working', ':', '(', 'thanks', '!']
Policy keeping negations: ['good', 'morning', ',', 'phone', 'not', 'working', ':', '(', 'thanks', '!']
Removing negations can turn 'not good' or 'never works' into positive-neutral ideas. Sentiment systems should keep not, no, and never unless negation is handled explicitly.


## Part F — Linguistic Analysis
### Task 11 — Part-of-speech tagging

In [13]:
docs = list(nlp.pipe(df["message"].astype(str), batch_size=50))
for i, doc in enumerate(docs[:5]):
    print(f"\nMessage {i+1} ({df.loc[i, 'message_id']}):")
    print([(t.text, t.pos_, t.tag_) for t in doc])

all_pos = Counter()
verbs = []
adjectives = []
for doc in docs:
    all_pos.update(t.pos_ for t in doc if not t.is_space and not t.is_punct)
    for t in doc:
        if t.pos_ == "VERB" and t.text.lower() not in verbs:
            verbs.append(t.text.lower())
        if t.pos_ == "ADJ" and t.text.lower() not in adjectives:
            adjectives.append(t.text.lower())

wanted = ["NOUN", "VERB", "ADJ", "ADV"]
pos_table = pd.DataFrame({"POS": wanted, "Count": [all_pos[p] for p in wanted]})
display(pos_table)
print("Five verbs:", verbs[:5])
print("Five adjectives:", adjectives[:5])
print("POS tagging identifies the grammatical role of each word. It helps analyze sentence structure, select specific word classes, and improve lemmatization with better grammatical context.")


Message 1 (MSG001):
[('Good', 'ADJ', 'JJ'), ('morning', 'NOUN', 'NN'), (',', 'PUNCT', ','), ('My', 'PRON', 'PRP$'), ('phone', 'NOUN', 'NN'), ('is', 'AUX', 'VBZ'), ('not', 'PART', 'RB'), ('working', 'VERB', 'VBG'), (':(', 'PUNCT', 'NFP'), (' ', 'SPACE', '_SP'), ('Thanks', 'NOUN', 'NNS'), ('again', 'ADV', 'RB'), ('!', 'PUNCT', '.')]

Message 2 (MSG002):
[('Hi', 'INTJ', 'UH'), (',', 'PUNCT', ','), ('Thanks', 'NOUN', 'NNS'), ('for', 'ADP', 'IN'), ('your', 'PRON', 'PRP$'), ('quick', 'ADJ', 'JJ'), ('response', 'NOUN', 'NN'), ('!', 'PUNCT', '.'), ('I', 'PRON', 'PRP'), ('need', 'VERB', 'VBP'), ('help', 'NOUN', 'NN'), ('ASAP', 'ADV', 'RB'), ('!', 'PUNCT', '.'), ('!', 'PUNCT', '.'), ('!', 'PUNCT', '.')]

Message 3 (MSG003):
[('Hi', 'INTJ', 'UH'), ('support', 'NOUN', 'NN'), (',', 'PUNCT', ','), ('I', 'PRON', 'PRP'), ('LOVE', 'VERB', 'VBP'), ('this', 'DET', 'DT'), ('product', 'NOUN', 'NN'), ('😍', 'NOUN', 'NN'), ('😍', 'PROPN', 'NNP'), ('Can', 'AUX', 'MD'), ('someone', 'PRON', 'NN'), ('help', 'VERB

,POS,Count
0,NOUN,245
1,VERB,179
2,ADJ,102
3,ADV,52


Five verbs: ['working', 'need', 'love', 'help', 'check']
Five adjectives: ['good', 'quick', 'new', 'amazing', 'dear']
POS tagging identifies the grammatical role of each word. It helps analyze sentence structure, select specific word classes, and improve lemmatization with better grammatical context.


### Task 12 — Named Entity Recognition (NER)

In [14]:
# Inspect entities in the first 15 original messages.
for i, doc in enumerate(docs[:15]):
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    print(df.loc[i, "message_id"], entities)

entity_rows = []
for i, doc in enumerate(docs):
    for ent in doc.ents:
        entity_rows.append({"message_id": df.loc[i, "message_id"], "entity_text": ent.text, "entity_label": ent.label_})
entities_df = pd.DataFrame(entity_rows, columns=["message_id", "entity_text", "entity_label"])
display(entities_df.head(20))
print("Entity counts:")
print(entities_df["entity_label"].value_counts())
print("Manual inspection should record expected entities found, missed, or incorrectly labeled. No entity does not mean the message has an error.")
print("A general spaCy NER model does not guarantee detection of an email address or order number. These formats may not match its training categories; regex is more reliable for explicit structured values.")

MSG001 []
MSG002 []
MSG003 []
MSG004 []
MSG005 [('250', 'CARDINAL'), ('EGP', 'ORG'), ('01010011705', 'DATE')]
MSG006 [('10102', 'DATE')]
MSG007 [('@support', 'GPE')]
MSG008 [('7 days', 'DATE')]
MSG009 []
MSG010 [('01010023410', 'DATE')]
MSG011 [('A-45-2026', 'DATE')]
MSG012 []
MSG013 [('2', 'CARDINAL'), ('1', 'CARDINAL')]
MSG014 [('#', 'CARDINAL'), ('@support', 'GPE')]
MSG015 [('01010035115', 'DATE')]


,message_id,entity_text,entity_label
0,MSG005,250,CARDINAL
1,MSG005,EGP,ORG
2,MSG005,01010011705,DATE
3,MSG006,10102,DATE
4,MSG007,@support,GPE
5,MSG008,7 days,DATE
6,MSG010,01010023410,DATE
7,MSG011,A-45-2026,DATE
8,MSG013,2,CARDINAL
9,MSG013,1,CARDINAL


Entity counts:
entity_label
DATE        43
CARDINAL    19
GPE          9
ORG          6
PERSON       6
Name: count, dtype: int64
Manual inspection should record expected entities found, missed, or incorrectly labeled. No entity does not mean the message has an error.
A general spaCy NER model does not guarantee detection of an email address or order number. These formats may not match its training categories; regex is more reliable for explicit structured values.


### Task 13 — Spelling correction experiment

In [15]:
# TextBlob can be slow on all 80 rows, so inspect 5 selected messages.
def correct_spelling(text):
    return str(TextBlob(str(text)).correct())

sample_spelling = df.loc[df["message"].str.contains(r"reciev|mssing|plese|defin|teh|wrok", case=False, regex=True, na=False)].head(5)
if sample_spelling.empty:
    sample_spelling = df.head(5)
sample_spelling = sample_spelling.copy()
sample_spelling["corrected_message"] = sample_spelling["clean_message"].apply(correct_spelling)
display(sample_spelling[["message_id", "clean_message", "corrected_message"]])
print("Automatic correction can change brand names, slang, names, emojis, placeholders, technical terms, or words where the intended meaning differs from the most common spelling.")

,message_id,clean_message,corrected_message
0,MSG001,"Good morning, My phone is not working :( Thank...","Good morning, By phone is not working :( Thank..."
1,MSG002,"Hi, Thanks for your quick response! I need hel...","I, Thanks for your quick response! I need help..."
2,MSG003,"Hi support, I LOVE this product 😍😍 Can someone...","I support, I LOVE this product 😍😍 An someone h..."
3,MSG004,"Hey there, The new update is AMAZING!! Please ...","Key there, The new update is AMAZING!! Please ..."
4,MSG005,"Dear team, The price is [NUMBER] EGP, too expe...","Dear team, The price is [NUMBER] EGP, too expe..."


Automatic correction can change brand names, slang, names, emojis, placeholders, technical terms, or words where the intended meaning differs from the most common spelling.


## Part H — Compare Pipelines and Final Report
### Task 14 — Build two preprocessing versions

In [16]:
def version_a(text):
    """Light preprocessing: cleaning, tokens, punctuation and negation kept."""
    return nltk_words(clean_text(text))

def version_b(text):
    """Strong preprocessing: clean, lowercase, remove NLTK stops, and lemmatize."""
    tokens = nltk_words(clean_text(text).lower())
    tokens = [t for t in tokens if t.lower() not in nltk_stop]
    return [lemmatizer.lemmatize(t) for t in tokens if t.isalpha()]

df["version_a_tokens"] = df["message"].apply(version_a)
df["version_b_tokens"] = df["message"].apply(version_b)
display(df[["message", "version_a_tokens", "version_b_tokens"]].head(10))

avg_a = df["version_a_tokens"].apply(len).mean()
avg_b = df["version_b_tokens"].apply(len).mean()
print(f"Average Version A token count: {avg_a:.2f}")
print(f"Average Version B token count: {avg_b:.2f}")
print("Information loss examples")
for i in range(3):
    print(f"Message {i+1}:", df.loc[i, "message"])
    print("A:", df.loc[i, "version_a_tokens"])
    print("B:", df.loc[i, "version_b_tokens"])
print("In Version B, punctuation, emojis, stop words, and case distinctions are removed. These can carry emphasis or emotional meaning, and negation words can change sentiment.")
print("Recommendation: For a first sentiment-analysis experiment, I would choose Version A (or Version B with negations kept) because it retains emotional emphasis and negation. Version B is useful for a simpler topic/classification baseline, not automatic removal of all sentiment cues.")

,message,version_a_tokens,version_b_tokens
0,"Good morning, My phone is not working :( Than...","[Good, morning, ,, My, phone, is, not, working...","[good, morning, phone, working, thanks]"
1,"Hi, Thanks for your quick response! I need hel...","[Hi, ,, Thanks, for, your, quick, response, !,...","[hi, thanks, quick, response, need, help, asap]"
2,"Hi support, I LOVE this product 😍😍 Can someone...","[Hi, support, ,, I, LOVE, this, product, 😍😍, C...","[hi, support, love, product, someone, help, em..."
3,"Hey there, The new update is AMAZING!! Please ...","[Hey, there, ,, The, new, update, is, AMAZING,...","[hey, new, update, amazing, please, check, vis..."
4,"Dear team, The price is 250 EGP, too expensive...","[Dear, team, ,, The, price, is, [, NUMBER, ], ...","[dear, team, price, number, egp, expensive, th..."
5,"Dear support, The package for order 10102 is d...","[Dear, support, ,, The, package, for, order, [...","[dear, support, package, order, number, damage..."
6,"Hello, Bad quality... very disappointed. What ...","[Hello, ,, Bad, quality, ..., very, disappoint...","[hello, bad, quality, disappointed, urgent, su..."
7,"Hello team, Great quality, but delivery took 7...","[Hello, team, ,, Great, quality, ,, but, deliv...","[hello, team, great, quality, delivery, took, ..."
8,"Good morning, Thanks! Great service :) Thanks ...","[Good, morning, ,, Thanks, !, Great, service, ...","[good, morning, thanks, great, service, thanks..."
9,"Hi, Can you change my delivery address? I need...","[Hi, ,, Can, you, change, my, delivery, addres...","[hi, change, delivery, address, need, help, as..."


Average Version A token count: 21.77
Average Version B token count: 9.44
Information loss examples
Message 1: Good morning, My phone is not working :(  Thanks again!
A: ['Good', 'morning', ',', 'My', 'phone', 'is', 'not', 'working', ':', '(', 'Thanks', 'again', '!']
B: ['good', 'morning', 'phone', 'working', 'thanks']
Message 2: Hi, Thanks for your quick response! I need help ASAP!!!
A: ['Hi', ',', 'Thanks', 'for', 'your', 'quick', 'response', '!', 'I', 'need', 'help', 'ASAP', '!', '!', '!']
B: ['hi', 'thanks', 'quick', 'response', 'need', 'help', 'asap']
Message 3: Hi support, I LOVE this product 😍😍 Can someone help me? Email: customer3@example.com
A: ['Hi', 'support', ',', 'I', 'LOVE', 'this', 'product', '😍😍', 'Can', 'someone', 'help', 'me', '?', 'Email', ':', '[', 'EMAIL', ']']
B: ['hi', 'support', 'love', 'product', 'someone', 'help', 'email', 'email']
In Version B, punctuation, emojis, stop words, and case distinctions are removed. These can carry emphasis or emotional meaning, an